In [2]:
import ee
import ee.mapclient
# Visualización en Jupyter Notebook usando folium y Earth Engine
# Instala folium y geemap si no están instalado

import folium
import geemap

In [3]:
# Name of project in google cloud available to use with Earth Engine
EE_PROJECT = "ee-gis-scode"

try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
except Exception as e:
    raise RuntimeError(
        "Failed to initialize the Earth Engine client. "
        "Make sure you have authenticated by running 'earthengine authenticate' "
        "in your command line. "
        "If you have already authenticated, ensure that the EE_PROJECT variable is set correctly."
    ) from e

In [9]:
# --- 1. Definir Área de Interés (AOI) ---
# Usaremos las coordenadas de Cuenca.
# Recommendation: Para un análisis más preciso, podrías cargar un shapefile.
xmin, ymin = -79.350128,-2.992413
xmax, ymax = -78.932648,-2.720840
cuenca_aoi = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])

# --- 2. Cargar el Dataset de Hansen Global Forest Change ---
# hansen_dataser: Proporciona un mapa ráster (una imagen, no un shapefile) donde cada píxel 
# indica el año en que se perdió la cobertura forestal.
# Se le pides a GEE que dé acceso a todo el dataset.
hansen_dataset = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
print("Dataset de Hansen cargado.")

Dataset de Hansen cargado.


In [14]:
# --- 3. Seleccionar las Bandas de Interés ---
# Extraemos las dos "capas" que nos importan de la super-imagen.
#treecover2000: Una capa que muestra el porcentaje de cobertura de árboles en el año 2000. La usaremos para definir dónde había bosque originalmente.
#lossyear: Una capa donde el valor de cada píxel indica el año en que se perdió el bosque (si es que se perdió). Un valor de 23 significa que la pérdida ocurrió en 2023.
tree_cover_2000 = hansen_dataset.select(['treecover2000'])
loss_year = hansen_dataset.select(['lossyear'])

In [15]:
# MÁSCARA 1: Zonas Deforestadas en 2025 (Label = 1)
# Le decimos a GEE: "Encuentra todos los píxeles en la banda 'loss_year'
# donde el valor sea exactamente 23 (el año 2025)".
# El resultado es una imagen donde esos píxeles valen 1 y el resto 0.
deforested_2023 = loss_year.eq(24) # eq() significa "equals"

In [16]:
# MÁSCARA 2: Zonas de Bosque Estable (Label = 0)
# La lógica es un poco más compleja:
# a. Píxeles que tenían más de 30% de cobertura de árboles en el 2000.
# b. Y... donde NUNCA hubo pérdida (la banda 'lossyear' es igual a 0).
stable_forest = tree_cover_2000.gte(30).And(loss_year.eq(0)) # gte() significa "greater than or equal"

In [17]:
# --- 5. Visualización Detallada de la Cobertura ---
# Parámetros de visualización mejorados
palette_deforested = ['FF0000'] # Solo Rojo para Deforestado (Valor 1)
palette_forest     = ['008000'] # Solo Verde Oscuro para Bosque Estable (Valor 1)

# Creamos una paleta para el tercer estado: TIERRA NO FORESTAL.
# Este color se aplica a todos los píxeles (0) que no son ni deforestación ni bosque estable.
palette_background = ['AAAAAA'] # Gris para Áreas No Forestales (Ríos, Cultivos, Urbano)

# --- 1. CONFIGURACIÓN DEL MAPA ---

# Calcular el centroide del AOI
centroid_coords = [ (xmin + xmax) / 2, (ymin + ymax) / 2 ]

# Crear un mapa centrado en el AOI
Map = geemap.Map(center=centroid_coords[::-1], zoom=10)
Map.setOptions('HYBRID') # Opcional: Mostrar etiquetas y límites de Google para mejor contexto

# --- 2. CREACIÓN DE LA IMAGEN DE BASE (Opcional, pero recomendado) ---

# Para visualizar las áreas NO-Bosque, crearemos una imagen que representa todo el AOI
# con un color de fondo (gris). Las otras capas se superpondrán.

# Creamos una imagen binaria simple donde todo el AOI es 1, para pintarlo de gris.
background_mask = ee.Image(1).clip(cuenca_aoi).rename('NO_BOSQUE')

# --- 3. AÑADIR CAPAS AL MAPA (Orden de Capas Importa) ---

# Capa Base: NO-Bosque (Lo que no nos interesa)
Map.addLayer(
    background_mask,
    {'palette': palette_background, 'min': 0, 'max': 1},
    '1. Fondo (No Forestal)',
    opacity=0.5, # Hacemos el fondo semi-transparente
    shown=True
)

# Capa 2: Bosque Estable (Lo que queda y siempre fue bosque)
Map.addLayer(
    stable_forest.updateMask(stable_forest.clip(cuenca_aoi)), # Usar updateMask para forzar visibilidad
    {'palette': palette_forest, 'min': 1, 'max': 1},         # El min y max en 1 asegura que solo se pinta el valor "Bosque"
    '2. Bosque Estable (NO DEFORESTADO)',
    opacity=1.0,
    shown=True
)

# Capa 3: Deforestado (Lo más importante: se superpone a todo)
Map.addLayer(
    deforested_2023.updateMask(deforested_2023.clip(cuenca_aoi)), # Usar updateMask
    {'palette': palette_deforested, 'min': 1, 'max': 1},           # El min y max en 1 asegura que solo se pinta el valor "Deforestado"
    '3. Área DEFORESTADA 2023 (Pérdida)',
    opacity=1.0,
    shown=True
)

# --- 4. LEYENDA Y DISPLAY ---

Map.add_legend(title="Leyenda de Cobertura", legend_dict={
    "Bosque Estable": "#008000",
    "Área Deforestada": "#FF0000",
    "Otras Tierras/No Forestal": "#AAAAAA"
})

display(Map)

Map(center=[-2.8566265, -79.141388], controls=(WidgetControl(options=['position', 'transparent_bg'], position=…